In [1]:
# Pipeline Configuration
RUN_MODE = "test"  # "test" or "production"
TEST_SAMPLE_SIZE = 10
MAX_WORKERS = 4
CHUNK_SIZE = 500
MAX_RETRIES = 5
TIMEOUT = 30
ENABLE_CACHE = True
ENABLE_CHECKPOINT = True
SAVE_PREVIEW_IMAGES = True
PIPELINE_VERSION = "1.0.0"


# Download Raw Datasets
This notebook checks and downloads the required SoilGrids global rasters and reference ERA5 weather datasets if they are missing locally.

In [2]:
import os
import requests
from bs4 import BeautifulSoup

# Ensure raw directories exist
os.makedirs("../data/raw/soilgrids", exist_ok=True)
os.makedirs("../data/raw/era5", exist_ok=True)

layers = ["phh2o", "nitrogen", "soc", "clay", "sand", "silt"]
urls = {
    "phh2o": "https://files.isric.org/soilgrids/latest/data_aggregated/1000m/phh2o/phh2o_0-5cm_mean.tif",
    "nitrogen": "https://files.isric.org/soilgrids/latest/data_aggregated/1000m/nitrogen/nitrogen_0-5cm_mean.tif",
    "soc": "https://files.isric.org/soilgrids/latest/data_aggregated/1000m/soc/soc_0-5cm_mean.tif",
    "clay": "https://files.isric.org/soilgrids/latest/data_aggregated/1000m/clay/clay_0-5cm_mean.tif",
    "sand": "https://files.isric.org/soilgrids/latest/data_aggregated/1000m/sand/sand_0-5cm_mean.tif",
    "silt": "https://files.isric.org/soilgrids/latest/data_aggregated/1000m/silt/silt_0-5cm_mean.tif"
}

for layer in layers:
    filepath = f"../data/raw/soilgrids/{layer}.tif"
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1024*1024:
        print(f"{layer}.tif already exists ✔ (skipping download)")
    else:
        print(f"Downloading {layer}.tif...")
        url = urls[layer]
        for attempt in range(MAX_RETRIES):
            try:
                r = requests.get(url, stream=True, timeout=TIMEOUT)
                r.raise_for_status()
                with open(filepath, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Successfully downloaded {layer}.tif")
                break
            except Exception as e:
                print(f"Error downloading {layer}.tif: {e}. Retrying...")


phh2o.tif already exists ✔ (skipping download)
nitrogen.tif already exists ✔ (skipping download)
soc.tif already exists ✔ (skipping download)
clay.tif already exists ✔ (skipping download)
sand.tif already exists ✔ (skipping download)
silt.tif already exists ✔ (skipping download)


In [3]:
filepath = "../data/raw/era5/era5.nc"
if os.path.exists(filepath) and os.path.getsize(filepath) > 1024:
    print("era5.nc already exists ✔ (skipping download)")
else:
    print("era5.nc reference file is missing. Checking CDS credentials...")
    # If credentials are not setup, skip
    if not os.path.exists(os.path.expanduser("~/.cdsapirc")):
        print("CDS API credentials file not found. Skipping reference ERA5 download.")
    else:
        import cdsapi
        client = cdsapi.Client()
        try:
            client.retrieve(
                "reanalysis-era5-land",
                {
                    "variable": "2m_temperature",
                    "year": "2024",
                    "month": "05",
                    "day": "22",
                    "time": "12:00",
                    "data_format": "netcdf",
                    "download_format": "unarchived"
                },
                filepath
            )
            print("Reference era5.nc downloaded successfully.")
        except Exception as e:
            print(f"Could not download reference ERA5 netcdf: {e}")


era5.nc already exists ✔ (skipping download)
